In [1]:
from datetime import datetime, timedelta
import backtrader as bt
import pandas as pd

class MyStrategy(bt.Strategy):
    def __init__(self):
        # przypisanie feedów do zmiennych w strategii
        self.data_m1  = self.datas[0]  # pierwszy dodany feed
        self.data_m15 = self.datas[1]  # drugi feed
        self.data_h1 = self.datas[2]  # drugi feed
        self.data_h4  = self.datas[3]  # trzeci feed


        self.last_execution_order = datetime.now() - timedelta(days=2000)
        self.STOP_LOSS_DIFF = 0.1
        self.TAKE_PROFIT_DIFF = 0.1
        self.DEFAULT_POSITION_SIZE = 50
        self.start_price = None
        self.START_HOUR = 8
        self.END_HOUR = 17
        self.POSITION_BRAKE_DELTA = timedelta(minutes=15)


    def next(self):
        last10_h4 = list(self.data_h4.close.get(size=10))
        if self.can_open_new_position() and len(last10_h4) > 2:
            last10_m1 = list(self.data_m1.close.get(size=10))
            last10_m15 = list(self.data_m15.close.get(size=10))
            last10_h4 = list(self.data_h4.close.get(size=10))
            if last10_h4[0] > last10_h4[1] > last10_h4[2]:
                if last10_m1[0] > last10_m1[1] > last10_m1[2]:
                    self.buy(size=self.DEFAULT_POSITION_SIZE)
            elif last10_h4[0] < last10_h4[1] < last10_h4[2]:
                if last10_m1[0] < last10_m1[1] < last10_m1[2]:
                    self.sell(size=self.DEFAULT_POSITION_SIZE)

        # if self.can_open_new_position():
        #     print("OPEN NEW POSITION")

            # print("XXXXXZZZZ")
            # print(self.position.price)
            # self.sell(exectype=bt.Order.Stop, price=self.data_m1.close[0] - 0.1)
            # self.buy_bracket(size=50, stopprice=self.data_m1.close[0] - 10)
            # print(self.data_m1.close[0])

        # target_close_date = datetime(2025, 10, 24, 12, 9, 0)
        # if self.position and current_date == target_close_date:
        #     print(self.position.size)
        #     print(self.position.price)
        #     print(self.broker.orders)
        #     print([x.getstatusname() for x in self.broker.orders])
        #     print([x.price for x in self.broker.orders])
        #     # self.close()

    def notify_order(self, order):
        if order.status == order.Completed:
            size = self.position.size
            executed_size = order.executed.size
            executed_price = order.executed.price
            if size == 0:
                print("Order Completed = position = 0")
                for o in self.broker.get_orders_open():
                    self.cancel(o)
                self.last_execution_order = self._get_current_date()

            if size == executed_size:
                self.start_price = executed_price
                print(self._get_current_date())
                if size > 0:
                    print(f"Order Completed = position > 0 = {size}")
                    self.sell(exectype=bt.Order.Stop, price=executed_price - self.STOP_LOSS_DIFF, size=executed_size)
                    self.sell(exectype=bt.Order.Limit, price=executed_price + self.TAKE_PROFIT_DIFF, size=executed_size)
                elif size < 0:
                    print(f"Order Completed = position < 0 = {size}")
                    self.buy(exectype=bt.Order.Stop, price=executed_price + self.STOP_LOSS_DIFF, size=executed_size)
                    self.buy(exectype=bt.Order.Limit, price=executed_price - self.TAKE_PROFIT_DIFF, size=executed_size)

    def can_open_new_position(self):
        current_date = self._get_current_date()
        is_hour_right = current_date.hour >= self.START_HOUR and current_date.hour <= self.END_HOUR
        was_last_position_a_while_ago = current_date - self.POSITION_BRAKE_DELTA > self.last_execution_order
        return self.position.size == 0 and is_hour_right and was_last_position_a_while_ago

    def _get_current_date(self):
        return bt.num2date(self.data_m1.datetime[-1])

In [2]:
class SuperTrend(bt.Indicator):
    """
    SuperTrend Indicator
    """
    plotinfo = dict(subplot=False) # Plot on the main price chart
    lines = ('supertrend',) # Define the single output line
    params = (('period', 10), ('multiplier', 3.0),) # Customizable parameters

    def __init__(self):
        # ATR and Median Price are needed for the calculation
        self.atr = bt.indicators.AverageTrueRange(period=self.p.period)
        self.median_price = (self.data.high + self.data.low) / 2.0

        # These bands are dynamic based on ATR and median price
        self.basic_upper_band = self.median_price + (self.p.multiplier * self.atr)
        self.basic_lower_band = self.median_price - (self.p.multiplier * self.atr)

    def next(self):
        # On the first bar, initialize SuperTrend with the close price
        if len(self) == 1:
            self.lines.supertrend[0] = self.data.close[0]
            return

        prev_st = self.lines.supertrend[-1]
        prev_close = self.data.close[-1]

        # --- Calculate the current SuperTrend value ---
        # If the previous trend was UP (previous close > previous SuperTrend)
        if prev_close > prev_st:
            # The new ST is the max of the previous ST and the current lower band
            self.lines.supertrend[0] = max(self.basic_lower_band[0], prev_st)
        else: # If the previous trend was DOWN
            # The new ST is the min of the previous ST and the current upper band
            self.lines.supertrend[0] = min(self.basic_upper_band[0], prev_st)

        # --- Check for a flip in the trend and adjust the SuperTrend line ---
        current_close = self.data.close[0]
        if current_close > self.lines.supertrend[0]: # If price is now above the ST line
            # We are in an uptrend, so the ST line should be based on the lower band
            self.lines.supertrend[0] = self.basic_lower_band[0]
        elif current_close < self.lines.supertrend[0]: # If price is now below the ST line
            # We are in a downtrend, so the ST line should be based on the upper band
            self.lines.supertrend[0] = self.basic_upper_band[0]

In [3]:
from strategies.supertrend import SuperTrendStrategy

INITIAL_CASH = 100000


def load_csv(path):
    df = pd.read_csv(path)
    df["open"]  = df["low"] + df["delta_open"]
    df["close"] = df["low"] + df["delta_close"]
    df["high"]  = df["low"] + df["delta_high"]
    df['datetime'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('datetime').sort_index()
    df = df[["open", "high", "low", "close", "volume"]]
    return bt.feeds.PandasData(dataname=df)

def test_strategy(strategy):

    data_m1  = load_csv("data/XTIUSD_20250930_2200_20251031_2259_M1.csv")
    data_m15 = load_csv("data/XTIUSD_20250930_2200_20251031_2259_M15.csv")
    data_h1 = load_csv("data/XTIUSD_20250930_2200_20251031_2259_H1.csv")
    data_h4  = load_csv("data/XTIUSD_20250930_2200_20251031_2259_H4.csv")

    cerebro = bt.Cerebro()
    cerebro.adddata(data_m1)
    cerebro.adddata(data_m15)
    cerebro.adddata(data_h1)
    cerebro.adddata(data_h4)

    cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')


    cerebro.addstrategy(strategy, st_period=10,
    st_multiplier=3.0,
    trail_percent=0.05)
    cerebro.broker.setcash(INITIAL_CASH)

    return cerebro ,cerebro.run()

In [5]:
from strategies.supertrend import SuperTrendStrategy

INITIAL_CASH = 100000


def load_csv(path):
    df = pd.read_csv(path)
    df["open"]  = df["low"] + df["delta_open"]
    df["close"] = df["low"] + df["delta_close"]
    df["high"]  = df["low"] + df["delta_high"]
    df['datetime'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('datetime').sort_index()
    df = df[["open", "high", "low", "close", "volume"]]
    return bt.feeds.PandasData(dataname=df)

def test_strategy_params(strategy, st_period, st_multiplier, trail_percent):

    data_m1  = load_csv("data/XTIUSD_20250930_2200_20251031_2259_M1.csv")
    data_m15 = load_csv("data/XTIUSD_20250930_2200_20251031_2259_M15.csv")
    data_h1 = load_csv("data/XTIUSD_20250930_2200_20251031_2259_H1.csv")
    data_h4  = load_csv("data/XTIUSD_20250930_2200_20251031_2259_H4.csv")

    cerebro = bt.Cerebro()
    cerebro.adddata(data_m1)
    cerebro.adddata(data_m15)
    cerebro.adddata(data_h1)
    cerebro.adddata(data_h4)

    cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')


    cerebro.addstrategy(strategy,
                        st_period=st_period,
                        st_multiplier=st_multiplier,
                        trail_percent=trail_percent)
    cerebro.broker.setcash(INITIAL_CASH)

    return cerebro ,cerebro.run()

In [4]:
cerebro, results = test_strategy(SuperTrendStrategy)
print("Starting Portfolio Value:", INITIAL_CASH)
print("Final Portfolio Value:", cerebro.broker.getvalue())
print(f"Profit: {cerebro.broker.getvalue() - INITIAL_CASH:.2f}$")
# === 5. Dostęp do analiz ===
strat = results[0]
analyzers = strat.analyzers

if hasattr(analyzers, 'trades'):
    print("Number of trades:", analyzers.trades.get_analysis().get('total', 0))

if hasattr(analyzers, 'sharpe'):
    print("Sharpe Ratio:", analyzers.sharpe.get_analysis())


Starting Portfolio Value: 100000
Final Portfolio Value: 100001.74450000002
Profit: 1.74$
Number of trades: AutoOrderedDict({'total': 3, 'open': 1, 'closed': 2})


In [9]:
def safe_get_total_trades(trade_analysis):
    # trade_analysis może mieć różne kształty, próbujemy wydobyć liczbę bez błędów
    if not trade_analysis:
        return 0
    # struktura zwykle: {'total': {'total': N, ...}, 'won': {...}, 'lost': {...}, ...}
    if isinstance(trade_analysis, dict):
        tot = trade_analysis.get('total')
        if isinstance(tot, dict):
            return tot.get('total', 0)
        # czasem analyzer zwraca inne pola — spróbuj bezpośrednio 'total'
        return trade_analysis.get('total', 0)
    return 0

st_periods = [5, 7, 10, 14]
st_multipliers = [2.0, 3.0, 4.0]
trail_percents = [0.02, 0.05, 0.07]

records = []

for p in st_periods:
    for m in st_multipliers:
        for t in trail_percents:
            print(f"Testing st_period={p}, st_multiplier={m}, trail_percent={t} ...")
            cerebro, results = test_strategy_params(SuperTrendStrategy, st_period=p,
                                                   st_multiplier=m, trail_percent=t)
            # Final portfolio value
            final_value = cerebro.broker.getvalue()
            profit = final_value - INITIAL_CASH

            # analizator
            strat = results[0]
            trades_an = getattr(strat.analyzers, 'trades', None)
            trades_count = 0
            if trades_an is not None:
                try:
                    ta = trades_an.get_analysis()
                    trades_count = safe_get_total_trades(ta)
                except Exception:
                    trades_count = 0

            print(f" -> Final value: {final_value:.2f}, Profit: {profit:.2f}, Trades: {trades_count}")

            records.append({
                'st_period': p,
                'st_multiplier': m,
                'trail_percent': t,
                'final_value': final_value,
                'profit': profit,
                'trades': trades_count
            })

df = pd.DataFrame(records)
df = df.sort_values(by='profit', ascending=False).reset_index(drop=True)


Testing st_period=5, st_multiplier=2.0, trail_percent=0.02 ...
 -> Final value: 99994.09, Profit: -5.91, Trades: 21
Testing st_period=5, st_multiplier=2.0, trail_percent=0.05 ...
 -> Final value: 100002.05, Profit: 2.05, Trades: 3
Testing st_period=5, st_multiplier=2.0, trail_percent=0.07 ...
 -> Final value: 99998.05, Profit: -1.95, Trades: 2
Testing st_period=5, st_multiplier=3.0, trail_percent=0.02 ...
 -> Final value: 99989.50, Profit: -10.50, Trades: 22
Testing st_period=5, st_multiplier=3.0, trail_percent=0.05 ...
 -> Final value: 100001.83, Profit: 1.83, Trades: 3
Testing st_period=5, st_multiplier=3.0, trail_percent=0.07 ...
 -> Final value: 99998.12, Profit: -1.88, Trades: 2
Testing st_period=5, st_multiplier=4.0, trail_percent=0.02 ...
 -> Final value: 99993.03, Profit: -6.97, Trades: 21
Testing st_period=5, st_multiplier=4.0, trail_percent=0.05 ...
 -> Final value: 100002.43, Profit: 2.43, Trades: 3
Testing st_period=5, st_multiplier=4.0, trail_percent=0.07 ...
 -> Final val

In [10]:
df

,st_period,st_multiplier,trail_percent,final_value,profit,trades
0,5,4.0,0.05,100002.4280,2.4280,3
1,7,4.0,0.05,100002.4280,2.4280,3
2,10,4.0,0.05,100002.4280,2.4280,3
3,7,2.0,0.05,100002.0480,2.0480,3
4,5,2.0,0.05,100002.0480,2.0480,3
5,14,4.0,0.05,100002.0380,2.0380,3
6,14,2.0,0.05,100002.0280,2.0280,3
7,10,2.0,0.05,100002.0280,2.0280,3
8,7,3.0,0.05,100001.8345,1.8345,3
9,5,3.0,0.05,100001.8345,1.8345,3


In [ ]:
# szukamy strategii otwierającej dużą liczbę pozycji -> i mającej dużą skuteczność
# przy takim samym stop loss i take profit skuteczność musi być ~80%